# Library management tool

This lab starts a tool to manage a library's loans, with SQLAlchemy's ORM on a SQLite database: model the books, the members and their loans, import data from a YAML file, query the database, then export the loans as CSV.

The steps build on each other: each solution defines what the next ones use. If you're stuck, run the solution before moving on.

## Setup

The next cell writes the library's data to `library.yaml`: books, members, and dated loans, some of them returned (they have a `returned` date). The one after creates the engine, on the `library.db` file, and the models' `Base` class.

In [ ]:
%%writefile library.yaml
books:
  - title: Vingt Mille Lieues sous les mers
    author: Jules Verne
  - title: Voyage au centre de la Terre
    author: Jules Verne
  - title: De la Terre à la Lune
    author: Jules Verne
  - title: Le Tour du monde en quatre-vingts jours
    author: Jules Verne
  - title: Michel Strogoff
    author: Jules Verne
  - title: The Hound of the Baskervilles
    author: Arthur Conan Doyle
  - title: Arsène Lupin, gentleman-cambrioleur
    author: Maurice Leblanc
members:
  - John Doe
  - Jane Doe
  - Sherlock Holmes
  - Arsène Lupin
loans:
  - book: Voyage au centre de la Terre
    member: Jane Doe
    date: "2026-01-03"
    returned: "2026-01-24"
  - book: Le Tour du monde en quatre-vingts jours
    member: Sherlock Holmes
    date: "2026-01-05"
    returned: "2026-01-19"
  - book: Michel Strogoff
    member: Sherlock Holmes
    date: "2026-01-07"
  - book: Arsène Lupin, gentleman-cambrioleur
    member: Jane Doe
    date: "2026-01-20"
    returned: "2026-02-01"
  - book: The Hound of the Baskervilles
    member: Sherlock Holmes
    date: "2026-02-02"
  - book: Voyage au centre de la Terre
    member: John Doe
    date: "2026-02-10"

In [ ]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Session

engine = create_engine("sqlite:///library.db")


class Base(DeclarativeBase):
  pass

## 1. Modeling books, members and loans

A member borrows several books, and a book is borrowed by several members over time: it's a many-to-many relationship. Each loan also carries information of its own, its date and its return date: it becomes a class in its own right, an *association object*. Describe, on the `Base` provided:

- `Book` (table `books`): `id`; `title`, unique; `author`; `loans`, the list of its loans;
- `Member` (table `members`): `id`; `name`, unique; `loans`;
- `Loan` (table `loans`): `id`; `id_book` and `id_member`, foreign keys; `loan_date`, a date; `return_date`, an optional date (an ongoing loan has none); `book` and `member`, the object on each side.

Drop the existing tables, then create these ones. Why does a loan have its own `id`, rather than a primary key made of `id_book` and `id_member`?

**Useful documentation**

- [Association object](https://docs.sqlalchemy.org/en/21/orm/basic_relationships.html#association-object)

In [ ]:
# Your code here

### Additional hints

A date is annotated `Mapped[date]` (`from datetime import date`), an optional column `Mapped[date | None]`, and `mapped_column(unique=True)` forbids duplicates. There are two relationships, each with its two sides, which name each other with `back_populates`: `Book.loans` and `Loan.book`, `Member.loans` and `Loan.member`.

```python
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship


class Book(Base):
  __tablename__ = "books"
  id: Mapped[int] = mapped_column(primary_key=True)
  title: Mapped[str] = mapped_column(unique=True)
  author: Mapped[str]
  loans: Mapped[list["Loan"]] = relationship(back_populates="book")


class Member(Base):
  ...


class Loan(Base):
  __tablename__ = "loans"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_book: Mapped[int] = mapped_column(ForeignKey(...))
  ...
  book: Mapped[Book] = relationship(back_populates=...)
  ...


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
```

### Solution

In [ ]:
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Book(Base):
  __tablename__ = "books"
  id: Mapped[int] = mapped_column(primary_key=True)
  title: Mapped[str] = mapped_column(unique=True)
  author: Mapped[str]
  loans: Mapped[list["Loan"]] = relationship(back_populates="book")


class Member(Base):
  __tablename__ = "members"
  id: Mapped[int] = mapped_column(primary_key=True)
  name: Mapped[str] = mapped_column(unique=True)
  loans: Mapped[list["Loan"]] = relationship(back_populates="member")


class Loan(Base):
  __tablename__ = "loans"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_book: Mapped[int] = mapped_column(ForeignKey("books.id"))
  id_member: Mapped[int] = mapped_column(ForeignKey("members.id"))
  loan_date: Mapped[date]
  return_date: Mapped[date | None]
  book: Mapped[Book] = relationship(back_populates="loans")
  member: Mapped[Member] = relationship(back_populates="loans")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# The solution starts from a new Base: declaring a Book class a second time on
# the same Base (if you already did above) would raise an error.
#
# A member can borrow the same book several times: the pair (id_book, id_member)
# doesn't identify a loan, so it can't be the primary key. Hence an id of each
# loan's own.

## 2. Importing the YAML file

Complete the function `import_yaml(path, session)`, which reads the YAML file `path` and adds to the session a `Book` per book, a `Member` per member and a `Loan` per loan. Loans refer to books and members by their title and name: keep the objects you create in dictionaries to find them. Then call it on `library.yaml` in a session, commit, and print the number of books, members and loans in the database.

**Useful documentation**

- [`yaml.safe_load`](https://pyyaml.org/wiki/PyYAMLDocumentation)
- [`date.fromisoformat`](https://docs.python.org/3/library/datetime.html#datetime.date.fromisoformat)

In [ ]:
def import_yaml(path: str, session: Session) -> None:
  """Add the books, members and loans of the file at path to the session."""
  ...

In [ ]:
# Your code here

### Additional hints

`yaml.safe_load(f)` returns the file's contents: a dictionary of lists. The dates are quoted in the file: they are strings, which `date.fromisoformat` converts, and `loan.get("returned")` is `None` for an ongoing loan. Adding the books, the members and the loans to the session is enough: the relationships link them. You should get 7 books, 4 members and 6 loans.

```python
import yaml


def import_yaml(path: str, session: Session) -> None:
  with open(path, encoding="utf-8") as f:
    data = yaml.safe_load(f)
  books = {
    book["title"]: Book(title=book["title"], author=book["author"])
    for book in data["books"]
  }
  members = {name: Member(name=name) for name in data["members"]}
  session.add_all(books.values())
  session.add_all(members.values())
  for loan in data["loans"]:
    returned = loan.get("returned")
    session.add(
      Loan(
        book=...,
        member=...,
        loan_date=...,
        return_date=...,
      )
    )


with Session(engine) as session:
  import_yaml("library.yaml", session)
  session.commit()
  for model in (Book, Member, Loan):
    print(model.__name__, len(session.scalars(select(model)).all()))
```

### Solution

In [ ]:
import yaml


def import_yaml(path: str, session: Session) -> None:
  with open(path, encoding="utf-8") as f:
    data = yaml.safe_load(f)
  books = {
    book["title"]: Book(title=book["title"], author=book["author"])
    for book in data["books"]
  }
  members = {name: Member(name=name) for name in data["members"]}
  session.add_all(books.values())
  session.add_all(members.values())
  for loan in data["loans"]:
    returned = loan.get("returned")
    session.add(
      Loan(
        book=books[loan["book"]],
        member=members[loan["member"]],
        loan_date=date.fromisoformat(loan["date"]),
        return_date=date.fromisoformat(returned) if returned else None,
      )
    )


with Session(engine) as session:
  import_yaml("library.yaml", session)
  session.commit()
  for model in (Book, Member, Loan):
    print(model.__name__, len(session.scalars(select(model)).all()))

## 3. Querying the database

Each query of this part is written with `select`, and run in a session with `session.scalars(...)`.

### The books Sherlock Holmes borrowed

Print the title of the books Sherlock Holmes borrowed, with each loan's date, sorted by date.

**Useful documentation**

- [`Select.order_by`](https://docs.sqlalchemy.org/en/21/tutorial/data_select.html#order-by)

In [ ]:
# Your code here

#### Additional hints

Start from `select(Loan)`, join `Loan.member`, filter on `Member.name` and sort by `Loan.loan_date` with `order_by`. Each loan gives its book through `loan.book`. You should get three books, borrowed from January 5 to February 2, 2026.

```python
with Session(engine) as session:
  query = select(Loan).join(...).where(...).order_by(...)
  for loan in session.scalars(query):
    print(loan.loan_date, loan.book.title)
```

#### Solution

In [ ]:
with Session(engine) as session:
  query = (
    select(Loan)
    .join(Loan.member)
    .where(Member.name == "Sherlock Holmes")
    .order_by(Loan.loan_date)
  )
  for loan in session.scalars(query):
    print(loan.loan_date, loan.book.title)

### The books never borrowed

Print, in alphabetical order, the title of the books that were never borrowed.

**Useful documentation**

- [`any`](https://docs.sqlalchemy.org/en/21/orm/internals.html#sqlalchemy.orm.RelationshipProperty.Comparator.any)

In [ ]:
# Your code here

#### Additional hints

`Book.loans.any()` is true for a book with at least one loan (in SQL, an `EXISTS`), and `~` negates it. You should get 2 books.

```python
with Session(engine) as session:
  query = select(Book).where(...).order_by(Book.title)
  for book in session.scalars(query):
    print(book.title)
```

#### Solution

In [ ]:
with Session(engine) as session:
  query = select(Book).where(~Book.loans.any()).order_by(Book.title)
  for book in session.scalars(query):
    print(book.title)

### The ongoing loans

Print the ongoing loans, those without a return date, by loan date: the date, the book's title and the member's name.

**Useful documentation**

- [`is_`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnOperators.is_)

In [ ]:
# Your code here

#### Additional hints

In SQL, a missing value is tested with `IS NULL`, which SQLAlchemy writes `Loan.return_date.is_(None)`. You should get 3 loans.

```python
with Session(engine) as session:
  query = select(Loan).where(...).order_by(Loan.loan_date)
  for loan in session.scalars(query):
    print(...)
```

#### Solution

In [ ]:
with Session(engine) as session:
  ongoing = Loan.return_date.is_(None)
  query = select(Loan).where(ongoing).order_by(Loan.loan_date)
  for loan in session.scalars(query):
    print(loan.loan_date, loan.book.title, loan.member.name)

## 4. Exporting the loans as CSV

Complete the function `export_csv(path, session)`, which writes every loan, by loan date, to the CSV file `path`: one row per loan, with the columns `book`, `member`, `loan_date` and `return_date` (empty for an ongoing loan). Call it to write `loans.csv`, then print the file's contents.

**Useful documentation**

- [`csv.DictWriter`](https://docs.python.org/3/library/csv.html#csv.DictWriter)
- [`date.isoformat`](https://docs.python.org/3/library/datetime.html#datetime.date.isoformat)

In [ ]:
def export_csv(path: str, session: Session) -> None:
  """Write the loans of the database to the CSV file at path."""
  ...

In [ ]:
# Your code here

### Additional hints

`csv.DictWriter(f, fieldnames=[...])` writes dictionaries: `writeheader()` writes the header, then `writerow(...)` each row. Open the file with `newline=""`, as the `csv` module requires. `isoformat()` writes a date as `2026-01-03`. The file should have 7 lines: the header and 6 loans.

```python
import csv


def export_csv(path: str, session: Session) -> None:
  with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["book", "member", "loan_date", "return_date"]
    )
    writer.writeheader()
    for loan in session.scalars(select(Loan).order_by(Loan.loan_date)):
      writer.writerow({
        "book": loan.book.title,
        "member": ...,
        "loan_date": ...,
        "return_date": ...,
      })


with Session(engine) as session:
  export_csv("loans.csv", session)

with open("loans.csv", encoding="utf-8") as f:
  print(f.read())
```

### Solution

In [ ]:
import csv


def export_csv(path: str, session: Session) -> None:
  with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["book", "member", "loan_date", "return_date"]
    )
    writer.writeheader()
    for loan in session.scalars(select(Loan).order_by(Loan.loan_date)):
      returned = loan.return_date
      writer.writerow(
        {
          "book": loan.book.title,
          "member": loan.member.name,
          "loan_date": loan.loan_date.isoformat(),
          "return_date": returned.isoformat() if returned else "",
        }
      )


with Session(engine) as session:
  export_csv("loans.csv", session)

with open("loans.csv", encoding="utf-8") as f:
  print(f.read())